# Testing
In this notebook, we do some minimal benchmarking of the DMPNN implementation on a synthetically generated dataset and compare to the GINEConv model. 

In [1]:
import os
import random
import time
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.datasets import MoleculeNet

os.chdir("..")

from examples.synthetic_graph_gen import generate_unique_graphs
from dmpnn.model import DMPNN
from dmpnn.training import DMPNNTrainer
from dmpnn.adapters import from_pyg_dataset

/Users/michaelmontemurri/Desktop/Research/simple-dmpnn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Synthetic Dataset

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
SEEDS = [0, 1, 2]
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3

train_graphs, train_sigs = generate_unique_graphs(10000, min_nodes=3, max_nodes=10, seed=42)
val_graphs, _ = generate_unique_graphs(2000, min_nodes=3, max_nodes=10, existing_sigs=train_sigs, seed=43)
test_graphs, _ = generate_unique_graphs(2000, min_nodes=3, max_nodes=10, existing_sigs=train_sigs, seed=44)
y_true_test = torch.vstack([g["y"] for g in test_graphs])

print(f"Using device: {device}")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


Using device: mps
Train/val/test sizes: 10000, 2000, 2000


In [3]:
DMPNN_CONFIG = {
    "node_feat_dim": 4,
    "edge_feat_dim": 2,
    "hidden_dim": 128,
    "num_steps": 3,
    "head_hidden_size": 128,
    "output_size": 1,
}


def run_dmpnn_seed(seed):
    set_seed(seed)

    model = DMPNN(**DMPNN_CONFIG)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    trainer = DMPNNTrainer(
        model,
        optimizer=optimizer,
        loss_fn=torch.nn.MSELoss(),
        device=device,
    )

    start = time.perf_counter()
    trainer.fit(
        train_graphs,
        val_graphs,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=False,
    )
    fit_seconds = time.perf_counter() - start

    test_mse = trainer.evaluate(test_graphs, batch_size=BATCH_SIZE)
    test_preds = trainer.predict(test_graphs, batch_size=BATCH_SIZE)
    test_mae = torch.mean(torch.abs(test_preds - y_true_test)).item()

    return {
        "model": "DMPNN",
        "seed": seed,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "fit_seconds": fit_seconds,
        "param_count": count_parameters(trainer.model),
    }


dmpnn_results = [run_dmpnn_seed(seed) for seed in SEEDS]
pd.DataFrame(dmpnn_results)


,model,seed,test_mse,test_mae,fit_seconds,param_count
0,DMPNN,0,0.003050,0.040589,80.162079,67201
1,DMPNN,1,0.002720,0.039696,67.191467,67201
2,DMPNN,2,0.001947,0.031189,66.357810,67201


In [4]:
def to_pyg_data(graph):
    return Data(
        x=graph["X"],
        edge_index=graph["edge_index"],
        edge_attr=graph["B"],
        y=graph["y"],
    )


train_dataset = [to_pyg_data(g) for g in train_graphs]
val_dataset = [to_pyg_data(g) for g in val_graphs]
test_dataset = [to_pyg_data(g) for g in test_graphs]


GINE_TASK_TYPES = {"regression", "binary_classification", "multiclass_classification"}


class GINE(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels, num_layers, out_channels=1, dropout=0.0):
        super().__init__()
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.convs = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))

        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels),
        )

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        edge_attr, batch_vec = batch.edge_attr, batch.batch

        x = self.node_encoder(x)
        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
            x = torch.relu(x)
            x = self.dropout(x)

        g = global_add_pool(x, batch_vec)
        return self.head(g)


def prepare_gine_targets(y, y_hat, task_type="regression"):
    if task_type not in GINE_TASK_TYPES:
        raise ValueError(f"Unsupported task_type: {task_type}")

    if task_type in {"regression", "binary_classification"}:
        return y.float().view_as(y_hat)

    return y.view(-1).long()


def postprocess_gine_predictions(y_hat, task_type="regression"):
    if task_type == "regression":
        return y_hat
    if task_type == "binary_classification":
        return (torch.sigmoid(y_hat) >= 0.5).long()
    if task_type == "multiclass_classification":
        return torch.argmax(y_hat, dim=1)
    raise ValueError(f"Unsupported task_type: {task_type}")


def train_epoch_gine(model, loader, optimizer, loss_fn, task_type="regression"):
    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        y_hat = model(batch)
        y = prepare_gine_targets(batch.y, y_hat, task_type)
        loss = loss_fn(y_hat, y)
        loss.backward()
        optimizer.step()

        batch_graphs = batch.num_graphs
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs

    return total_loss / total_graphs


@torch.no_grad()
def evaluate_gine(model, loader, loss_fn, task_type="regression"):
    model.eval()
    total_loss = 0.0
    total_graphs = 0
    logits_or_values = []
    targets = []

    for batch in loader:
        batch = batch.to(device)
        y_hat = model(batch)
        y = prepare_gine_targets(batch.y, y_hat, task_type)
        loss = loss_fn(y_hat, y)

        batch_graphs = batch.num_graphs
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs
        logits_or_values.append(y_hat.cpu())
        targets.append(y.cpu())

    avg_loss = total_loss / total_graphs
    logits_or_values = torch.cat(logits_or_values, dim=0)
    targets = torch.cat(targets, dim=0)
    preds = postprocess_gine_predictions(logits_or_values, task_type)

    metrics = {"loss": avg_loss}
    if task_type == "regression":
        metrics["mse"] = avg_loss
        metrics["rmse"] = np.sqrt(avg_loss)
        metrics["mae"] = torch.mean(torch.abs(preds - targets)).item()
    elif task_type == "binary_classification":
        target_labels = targets.float().view_as(preds.float())
        metrics["accuracy"] = (preds.float() == target_labels).float().mean().item()
        metrics["positive_rate"] = preds.float().mean().item()
    else:
        target_labels = targets.view(-1)
        metrics["accuracy"] = (preds.view(-1) == target_labels).float().mean().item()

    return metrics, preds, targets, logits_or_values


@torch.no_grad()
def predict_gine(model, loader, task_type="regression", return_probabilities=False):
    model.eval()
    outputs = []

    for batch in loader:
        batch = batch.to(device)
        outputs.append(model(batch).cpu())

    outputs = torch.cat(outputs, dim=0)
    if return_probabilities and task_type == "binary_classification":
        return torch.sigmoid(outputs)
    if return_probabilities and task_type == "multiclass_classification":
        return torch.softmax(outputs, dim=1)
    return postprocess_gine_predictions(outputs, task_type)


GINE_CONFIG = {
    "in_channels": 4,
    "edge_dim": 2,
    "hidden_channels": 128,
    "num_layers": 3,
    "out_channels": 1,
    "dropout": 0.0,
}


def run_gine_seed(seed):
    set_seed(seed)

    model = GINE(**GINE_CONFIG).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = torch.nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    start = time.perf_counter()
    for _ in range(EPOCHS):
        train_epoch_gine(model, train_loader, optimizer, loss_fn, task_type="regression")
        evaluate_gine(model, val_loader, loss_fn, task_type="regression")
    fit_seconds = time.perf_counter() - start

    test_metrics, _, _, _ = evaluate_gine(model, test_loader, loss_fn, task_type="regression")

    return {
        "model": "GINE",
        "seed": seed,
        "test_mse": test_metrics["mse"],
        "test_mae": test_metrics["mae"],
        "fit_seconds": fit_seconds,
        "param_count": count_parameters(model),
    }


gine_results = [run_gine_seed(seed) for seed in SEEDS]
pd.DataFrame(gine_results)

,model,seed,test_mse,test_mae,fit_seconds,param_count
0,GINE,0,0.020260,0.113116,149.985986,117505
1,GINE,1,0.002510,0.037253,117.914677,117505
2,GINE,2,0.002064,0.032886,118.440387,117505


In [5]:
all_results = pd.concat(
    [pd.DataFrame(dmpnn_results), pd.DataFrame(gine_results)],
    ignore_index=True,
)

summary = all_results.groupby("model", as_index=False).agg(
    test_mse_mean=("test_mse", "mean"),
    test_mse_std=("test_mse", "std"),
    test_mae_mean=("test_mae", "mean"),
    test_mae_std=("test_mae", "std"),
    fit_seconds_mean=("fit_seconds", "mean"),
    fit_seconds_std=("fit_seconds", "std"),
    param_count=("param_count", "first"),
)

print("Per-seed results:")
display(all_results)

print("Summary across 3 seeds:")
display(summary.sort_values("test_mse_mean"))


Per-seed results:


,model,seed,test_mse,test_mae,fit_seconds,param_count
0,DMPNN,0,0.003050,0.040589,80.162079,67201
1,DMPNN,1,0.002720,0.039696,67.191467,67201
2,DMPNN,2,0.001947,0.031189,66.357810,67201
3,GINE,0,0.020260,0.113116,149.985986,117505
4,GINE,1,0.002510,0.037253,117.914677,117505
5,GINE,2,0.002064,0.032886,118.440387,117505


Summary across 3 seeds:


,model,test_mse_mean,test_mse_std,test_mae_mean,test_mae_std,fit_seconds_mean,fit_seconds_std,param_count
0,DMPNN,0.002572,0.000566,0.037158,0.005188,71.237119,7.740474,67201
1,GINE,0.008278,0.010379,0.061085,0.045113,128.780350,18.366500,117505


# MoleculeNet Regression Dataset: ESOL

In [6]:
# Use a standard molecular regression benchmark from PyG.
dataset = MoleculeNet(root="data/MoleculeNet", name="ESOL")

# Convert to the internal graph format used by the DMPNN.
graphs = from_pyg_dataset(dataset, task_type="regression")

num_graphs = len(graphs)
train_end = int(0.8 * num_graphs)
val_end = int(0.9 * num_graphs)

train_graphs = graphs[:train_end]
val_graphs = graphs[train_end:val_end]
test_graphs = graphs[val_end:]

node_feat_dim = train_graphs[0]["X"].shape[1]
edge_feat_dim = train_graphs[0]["B"].shape[1]

model = DMPNN(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = DMPNNTrainer(
    model,
    optimizer=optimizer,
    loss_fn=torch.nn.MSELoss(),
    device=device,
    task_type="regression",
)

history = trainer.fit(
    train_graphs,
    val_graphs,
    epochs=20,
    batch_size=32,
    verbose=True,
)

preds = trainer.predict(test_graphs, batch_size=32)
y_true = torch.vstack([g["y"] for g in test_graphs])
test_mse = trainer.evaluate(test_graphs, batch_size=32)
test_mae = torch.mean(torch.abs(preds - y_true)).item()
test_rmse = np.sqrt(test_mse)

print("\nDMPNN on PyG: ESOL Dataset")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print("First 10 DMPNN predictions:")
for i, (y_t, y_p) in enumerate(zip(y_true[:10], preds[:10])):
    print(f"  graph {i:>2}: true = {y_t.item():>6.2f} | pred = {y_p.item():>6.2f}")


# GINEConv regression using the shared task-aware helpers.
train_dataset = [to_pyg_data(g) for g in train_graphs]
val_dataset = [to_pyg_data(g) for g in val_graphs]
test_dataset = [to_pyg_data(g) for g in test_graphs]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

gine_model = GINE(
    in_channels=node_feat_dim,
    edge_dim=edge_feat_dim,
    hidden_channels=128,
    num_layers=3,
    out_channels=1,
    dropout=0.0,
).to(device)

gine_optimizer = torch.optim.Adam(gine_model.parameters(), lr=1e-3)
gine_loss_fn = torch.nn.MSELoss()

for epoch in range(20):
    train_loss = train_epoch_gine(
        gine_model,
        train_loader,
        gine_optimizer,
        gine_loss_fn,
        task_type="regression",
    )
    val_metrics, _, _, _ = evaluate_gine(
        gine_model,
        val_loader,
        gine_loss_fn,
        task_type="regression",
    )
    print(
        f"Epoch {epoch + 1}/20 | "
        f"GINE train_loss={train_loss:.4f} | "
        f"val_mse={val_metrics['mse']:.4f} | "
        f"val_rmse={val_metrics['rmse']:.4f} | "
        f"val_mae={val_metrics['mae']:.4f}"
    )

gine_test_metrics, gine_preds, gine_targets, _ = evaluate_gine(
    gine_model,
    test_loader,
    gine_loss_fn,
    task_type="regression",
)

print("\nGINEConv on ESOL")
print(f"Test MSE: {gine_test_metrics['mse']:.4f}")
print(f"Test RMSE: {gine_test_metrics['rmse']:.4f}")
print(f"Test MAE: {gine_test_metrics['mae']:.4f}")
print("First 10 GINE predictions:")
for i, (y_t, y_p) in enumerate(zip(gine_targets[:10], gine_preds[:10])):
    print(f"  graph {i:>2}: true = {y_t.item():>6.2f} | pred = {y_p.item():>6.2f}")

Epoch 1/20 | train_loss=13.6205 | val_loss=2.9437
Epoch 2/20 | train_loss=1.9030 | val_loss=2.1619
Epoch 3/20 | train_loss=1.7629 | val_loss=1.8735
Epoch 4/20 | train_loss=1.6807 | val_loss=1.9446
Epoch 5/20 | train_loss=1.8434 | val_loss=2.0207
Epoch 6/20 | train_loss=1.6791 | val_loss=1.9540
Epoch 7/20 | train_loss=1.6254 | val_loss=1.7151
Epoch 8/20 | train_loss=1.4456 | val_loss=2.4490
Epoch 9/20 | train_loss=1.4120 | val_loss=1.5537
Epoch 10/20 | train_loss=1.3443 | val_loss=1.9007
Epoch 11/20 | train_loss=1.2888 | val_loss=1.5497
Epoch 12/20 | train_loss=1.3090 | val_loss=1.7466
Epoch 13/20 | train_loss=1.2091 | val_loss=1.6355
Epoch 14/20 | train_loss=1.1648 | val_loss=1.4894
Epoch 15/20 | train_loss=1.3633 | val_loss=1.5197
Epoch 16/20 | train_loss=1.1303 | val_loss=1.7604
Epoch 17/20 | train_loss=1.0060 | val_loss=1.6108
Epoch 18/20 | train_loss=1.1079 | val_loss=1.5705
Epoch 19/20 | train_loss=1.0554 | val_loss=1.3386
Epoch 20/20 | train_loss=0.8634 | val_loss=1.7115

DMPNN o

In [7]:
print("ESOL Performance Summary (Regression):")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print("\nDMPNN on ESOL")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print("\nGINEConv on ESOL")
print(f"Test MSE: {gine_test_metrics['mse']:.4f}")
print(f"Test RMSE: {gine_test_metrics['rmse']:.4f}")
print(f"Test MAE: {gine_test_metrics['mae']:.4f}")

ESOL Performance Summary (Regression):
Train/val/test sizes: 902, 113, 113

DMPNN on ESOL
Test MSE: 1.1509
Test RMSE: 1.0728
Test MAE: 0.8432

GINEConv on ESOL
Test MSE: 1.5721
Test RMSE: 1.2538
Test MAE: 0.9694


# MoleculeNet Classification Dataset: BBBP

In [8]:
# Binary Classification Example with a standard PyG benchmark dataset.
# BBBP labels are treated as 0/1 binary targets for graph-level classification.

bbbp_dataset = MoleculeNet(root="data/MoleculeNet", name="BBBP")
bbbp_graphs = from_pyg_dataset(bbbp_dataset, task_type="binary_classification")

clean_graphs = []
for g in bbbp_graphs:
    y = g["y"]
    if not torch.isfinite(y).all():
        continue
    g = dict(g)
    g["y"] = (y > 0).float().view(1)
    clean_graphs.append(g)

num_graphs = len(clean_graphs)
train_end = int(0.8 * num_graphs)
val_end = int(0.9 * num_graphs)

train_graphs = clean_graphs[:train_end]
val_graphs = clean_graphs[train_end:val_end]
test_graphs = clean_graphs[val_end:]

node_feat_dim = train_graphs[0]["X"].shape[1]
edge_feat_dim = train_graphs[0]["B"].shape[1]

clf_model = DMPNN(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

clf_optimizer = torch.optim.Adam(clf_model.parameters(), lr=1e-3)
clf_trainer = DMPNNTrainer(
    clf_model,
    optimizer=clf_optimizer,
    loss_fn=torch.nn.BCEWithLogitsLoss(),
    device=device,
    task_type="binary_classification",
)

clf_history = clf_trainer.fit(
    train_graphs,
    val_graphs,
    epochs=20,
    batch_size=32,
    verbose=True,
)

# `predict()` returns hard 0/1 labels for binary classification.
pred_labels = clf_trainer.predict(test_graphs, batch_size=32).float()
y_true = torch.vstack([g["y"] for g in test_graphs]).float()

test_bce = clf_trainer.evaluate(test_graphs, batch_size=32)
accuracy = (pred_labels.view_as(y_true) == y_true).float().mean().item()
positive_rate = pred_labels.float().mean().item()

print("DMPNN binary classification example: BBBP")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print(f"Test BCE loss: {test_bce:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Predicted positive rate: {positive_rate:.4f}")
print("First 10 DMPNN binary predictions:")
for i, (y_t, y_p) in enumerate(zip(y_true[:10], pred_labels[:10])):
    print(f"  graph {i:>2}: true = {int(y_t.item())} | pred = {int(y_p.item())}")


# GINEConv binary classification on the same BBBP split.
bbbp_train_dataset = [to_pyg_data(g) for g in train_graphs]
bbbp_val_dataset = [to_pyg_data(g) for g in val_graphs]
bbbp_test_dataset = [to_pyg_data(g) for g in test_graphs]

bbbp_train_loader = DataLoader(bbbp_train_dataset, batch_size=32, shuffle=True)
bbbp_val_loader = DataLoader(bbbp_val_dataset, batch_size=32, shuffle=False)
bbbp_test_loader = DataLoader(bbbp_test_dataset, batch_size=32, shuffle=False)

gine_clf_model = GINE(
    in_channels=node_feat_dim,
    edge_dim=edge_feat_dim,
    hidden_channels=128,
    num_layers=3,
    out_channels=1,
    dropout=0.0,
).to(device)

gine_clf_optimizer = torch.optim.Adam(gine_clf_model.parameters(), lr=1e-3)
gine_clf_loss_fn = torch.nn.BCEWithLogitsLoss()

for epoch in range(20):
    gine_train_loss = train_epoch_gine(
        gine_clf_model,
        bbbp_train_loader,
        gine_clf_optimizer,
        gine_clf_loss_fn,
        task_type="binary_classification",
    )
    gine_val_metrics, _, _, _ = evaluate_gine(
        gine_clf_model,
        bbbp_val_loader,
        gine_clf_loss_fn,
        task_type="binary_classification",
    )
    print(
        f"Epoch {epoch + 1}/20 | "
        f"GINE train_loss={gine_train_loss:.4f} | "
        f"val_bce={gine_val_metrics['loss']:.4f} | "
        f"val_accuracy={gine_val_metrics['accuracy']:.4f}"
    )

gine_test_metrics, _, gine_targets, _ = evaluate_gine(
    gine_clf_model,
    bbbp_test_loader,
    gine_clf_loss_fn,
    task_type="binary_classification",
)
gine_pred_labels = predict_gine(
    gine_clf_model,
    bbbp_test_loader,
    task_type="binary_classification",
)
gine_pred_probs = predict_gine(
    gine_clf_model,
    bbbp_test_loader,
    task_type="binary_classification",
    return_probabilities=True,
)

print("\nGINEConv binary classification example: BBBP")
print(f"Test BCE loss: {gine_test_metrics['loss']:.4f}")
print(f"Test accuracy: {gine_test_metrics['accuracy']:.4f}")
print(f"Predicted positive rate: {gine_test_metrics['positive_rate']:.4f}")
print("First 10 GINE binary predictions:")
for i, (y_t, y_p, y_prob) in enumerate(zip(gine_targets[:10], gine_pred_labels[:10], gine_pred_probs[:10])):
    print(
        f"  graph {i:>2}: true = {int(y_t.item())} | "
        f"pred = {int(y_p.item())} | prob = {y_prob.item():.3f}"
    )

Epoch 1/20 | train_loss=1.5776 | val_loss=0.4623
Epoch 2/20 | train_loss=0.5795 | val_loss=0.1253
Epoch 3/20 | train_loss=0.5599 | val_loss=0.2381
Epoch 4/20 | train_loss=0.4995 | val_loss=0.2504
Epoch 5/20 | train_loss=0.4761 | val_loss=0.3765
Epoch 6/20 | train_loss=0.4803 | val_loss=0.2507
Epoch 7/20 | train_loss=0.4386 | val_loss=0.2814
Epoch 8/20 | train_loss=0.4650 | val_loss=0.1047
Epoch 9/20 | train_loss=0.4860 | val_loss=0.1932
Epoch 10/20 | train_loss=0.4188 | val_loss=0.1329
Epoch 11/20 | train_loss=0.4484 | val_loss=0.1368
Epoch 12/20 | train_loss=0.4362 | val_loss=0.3438
Epoch 13/20 | train_loss=0.4092 | val_loss=0.1121
Epoch 14/20 | train_loss=0.3882 | val_loss=0.2313
Epoch 15/20 | train_loss=0.4184 | val_loss=0.1457
Epoch 16/20 | train_loss=0.3762 | val_loss=0.1603
Epoch 17/20 | train_loss=0.3980 | val_loss=0.1435
Epoch 18/20 | train_loss=0.3705 | val_loss=0.1339
Epoch 19/20 | train_loss=0.3960 | val_loss=0.1945
Epoch 20/20 | train_loss=0.3979 | val_loss=0.1320
DMPNN bin

In [10]:
print("BBBP Performance Summary (Classification):")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")

print("DMPNN")
print(f"Test BCE loss: {test_bce:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Predicted positive rate: {positive_rate:.4f}")
print("GINEConv")
print(f"Test BCE loss: {gine_test_metrics['loss']:.4f}")
print(f"Test accuracy: {gine_test_metrics['accuracy']:.4f}")
print(f"Predicted positive rate: {gine_test_metrics['positive_rate']:.4f}")

BBBP Performance Summary (Classification):
Train/val/test sizes: 1631, 204, 204
DMPNN
Test BCE loss: 0.1484
Test accuracy: 0.9755
Predicted positive rate: 0.9755
GINEConv
Test BCE loss: 0.1922
Test accuracy: 0.9804
Predicted positive rate: 0.9804
